In [1]:
import numpy as np
from scipy.integrate import solve_ivp as sp_solve_ivp
from scipy.integrate import odeint
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from typing import List
device = 'cpu'
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'serif'

from ftnode.utils import set_global_seed
from ftnode.node import (
    FTNODE, FeluSigmoidMLP, GeluSigmoidMLP,)

import torchode

seed = 1234
set_global_seed(seed)

random_state = 67

[Seed] Deterministic mode enabled (may reduce speed).


## Generate Data

In [2]:
## Define system 
gamma = 50
def sigmoid(x,gamma=gamma):
    return 1 / (1+np.exp(-gamma*x))

eps = 0.02
q1, q2 = (0.08, 0.04)
b1 = 1-eps
b2 = 1-eps

def c1_in(x):
    return q1*(1-sigmoid(x-b1))

def c2_in(y):
    return q1*(1-sigmoid(y-b2))

def c1_out(y):
    return q2*(1-sigmoid(y-b2))

def c2_out(y):
    return q2

def two_tank_system(t,x,u):
    x1, x2 = x
    p, v = u
    x1= np.maximum(x1,0)
    x2 = np.maximum(x2,0)
    dx1dt = c1_in(x1)*(1-v)*p-c1_out(x2)*np.sqrt(x1)
    dx2dt = c2_in(x2)*v*p +c1_out(x2)*np.sqrt(x1)-q2*np.sqrt(x2)
    return np.hstack([dx1dt,dx2dt])


p_vals = np.linspace(0,1,101)
v_vals = np.linspace(0,1,101)

In [3]:
t_max = 200
n_colloc = 501

x0s = np.linspace(0,1,21)

In [4]:
p_train = p_vals[10:-10:10]
v_train = v_vals[10:-10:10]

In [5]:
Xs = [] 
Us = []
t = np.linspace(0,t_max, n_colloc)
for pi in tqdm(p_train):
    for vi in v_train:
        
        for x0 in zip(x0s, x0s):
            u = np.array([pi,vi])

            sol = sp_solve_ivp(
                two_tank_system,
                t_span = [0,t_max],
                y0 = np.array(x0),
                t_eval= np.linspace(0,t_max, n_colloc),
                args =(u,)
            )

            Xs.append(sol.y.T)
            Us.append((pi,vi))

Us = np.array(Us)
Xs = np.array(Xs)

  0%|          | 0/9 [00:00<?, ?it/s]

In [6]:
dXs = np.zeros_like(Xs)
T = t[np.newaxis,:,np.newaxis]
X_diff = Xs[:,2:,:] - Xs[:,:-2,:]
T_diff = T[:,2:,:] - T[:,:-2,:]

dXs[:,1:-1,:] = X_diff/T_diff
dXs[:,0,:] = (Xs[:,1,:] - Xs[:,0,:]) / (T[:,1,:] - T[:,0,:])
dXs[:,-1,:] = (Xs[:,-1,:] - Xs[:,-2,:]) / (T[:,-1,:] - T[:,-2,:])

In [7]:
dX_tensor = [
    torch.tensor(dxi,dtype=torch.float32,device=device) for dxi in dXs
]
X_tensor = [
    torch.tensor(xi,dtype=torch.float32,device=device) for xi in Xs
]
U_tensor = [
    torch.tensor(ui,dtype=torch.float32,device=device) for ui in Us
]

T_tensor = [
    torch.tensor(t,dtype=torch.float32, device=device) for _ in range(len(Xs))
]

In [8]:
class GradDataset(torch.utils.data.Dataset):
    def __init__(self, dX: List, X: List, T: List, U: List):
        self.dX = dX
        self.X = X
        self.T = T
        self.U = U

    def __len__(self):
        return len(self.dX)

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError(
                f"Index {idx} is out of bounds of dataset size: {len(self)}."
            )

        dXi = self.dX[idx]
        Xi = self.X[idx]
        ti = self.T[idx]
        ui = self.U[idx]

        return dXi, Xi, ti, ui

dataset = GradDataset(dX = dX_tensor,X = X_tensor, T = T_tensor, U = U_tensor)
# dataloader = torch.utils.data.DataLoader(dataset, batch_size=50, shuffle=True)

## Run $k$-Folds Cross-validation

In [9]:
k_folds = 10

In [10]:
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, SubsetRandomSampler
import time
import copy

In [11]:
avg_best_val_losses = []
avg_best_train_losses = []

In [12]:
# --- Configuration ---
k_folds = k_folds
n_epochs = 1000  
batch_size = 50 
learning_rate = 1e-2
print_every = 100
_precision = 4
random_state = random_state


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)

val_results = []
train_results = []

def get_fresh_model_components():
    f = FeluSigmoidMLP(dims=[2,10,10,2],lower_bound=-1, upper_bound=-0.1)
    g = GeluSigmoidMLP(dims=[4,10,10,2],lower_bound=0, upper_bound=1)
    model = FTNODE(f,g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            
            ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            u_func = lambda t: ui_expanded

            opt.zero_grad()
            
            dXi_pred = model_fold(ti, Xi, u_func)
            loss = loss_criteria(dXi, dXi_pred)
            
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                u_func = lambda t: ui_expanded

                dXi_pred = model_fold(ti, Xi, u_func)
                loss = loss_criteria(dXi, dXi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)
avg_best_val_losses, np.argmin(avg_best_val_losses)


--- FOLD 1/10 ---


Fold 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.3440e-02, Val Loss = 1.2933e-03, Time = 5.8255e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 5.7782e-04, Val Loss = 2.8128e-04, Time = 3.5791e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.8160e-04, Val Loss = 1.0537e-04, Time = 3.3796e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 8.5729e-05, Val Loss = 6.1099e-05, Time = 3.4042e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 5.3325e-05, Val Loss = 3.9408e-05, Time = 3.5372e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 3.8839e-05, Val Loss = 2.9582e-05, Time = 3.0683e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 7.6639e-06, Val Loss = 7.7430e-06, Time = 2.5778e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.3514e-06, Val Loss = 2.9040e-06, Time = 3.0179e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 3.1533e-06, Val Loss = 3.2589e-06, Time = 3.2862e-01, lr = 3.1250e-04
Epoch 400: Train Loss = 3.1369e-06, Val Loss = 3.0321e-06, Time = 2.6007e-01, lr = 1.2207e-06
Epoch 500: Train Loss = 3.1347e-06, Val Loss = 2.9600e-06, Time = 2.7856

Fold 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.5291e-02, Val Loss = 1.4103e-03, Time = 2.6553e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 4.7634e-04, Val Loss = 2.2809e-04, Time = 2.8780e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.4952e-04, Val Loss = 1.1884e-04, Time = 2.8061e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 9.1517e-05, Val Loss = 8.6151e-05, Time = 2.6686e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 7.0239e-05, Val Loss = 6.5065e-05, Time = 2.6252e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 5.9194e-05, Val Loss = 5.9092e-05, Time = 2.6349e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 7.0954e-06, Val Loss = 7.7827e-06, Time = 2.6161e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.5105e-06, Val Loss = 3.9420e-06, Time = 2.5931e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 3.0401e-06, Val Loss = 3.0562e-06, Time = 2.5923e-01, lr = 7.8125e-05
Epoch 400: Train Loss = 3.0446e-06, Val Loss = 3.0978e-06, Time = 2.6124e-01, lr = 1.5259e-07
Epoch 500: Train Loss = 3.0435e-06, Val Loss = 3.2013e-06, Time = 2.7894

...

Fold 9:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 2.2411e-02, Val Loss = 4.2566e-03, Time = 3.4967e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.1823e-03, Val Loss = 3.4390e-04, Time = 3.2540e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 2.3132e-04, Val Loss = 1.6600e-04, Time = 3.3151e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 1.2775e-04, Val Loss = 9.5303e-05, Time = 3.3407e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 7.6832e-05, Val Loss = 5.6652e-05, Time = 3.1091e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 4.7974e-05, Val Loss = 3.7663e-05, Time = 3.0384e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 1.0093e-05, Val Loss = 9.4046e-06, Time = 3.3090e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 5.8359e-06, Val Loss = 5.2749e-06, Time = 2.9946e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 3.4926e-06, Val Loss = 3.0196e-06, Time = 3.2651e-01, lr = 1.0000e-02
Epoch 400: Train Loss = 3.1460e-06, Val Loss = 2.7633e-06, Time = 3.1897e-01, lr = 1.2500e-03
Epoch 500: Train Loss = 3.1023e-06, Val Loss = 2.7950e-06, Time = 2.8244

Fold 10:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.6821e-02, Val Loss = 1.2154e-03, Time = 3.1569e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 4.7253e-04, Val Loss = 2.0993e-04, Time = 3.0472e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.5750e-04, Val Loss = 1.1436e-04, Time = 3.0026e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 9.9081e-05, Val Loss = 7.5978e-05, Time = 3.0223e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 7.2958e-05, Val Loss = 6.1853e-05, Time = 3.0629e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 5.8703e-05, Val Loss = 4.8747e-05, Time = 2.9502e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 7.1279e-06, Val Loss = 6.9094e-06, Time = 2.8418e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.2880e-06, Val Loss = 3.3401e-06, Time = 2.8708e-01, lr = 5.0000e-03
Epoch 300: Train Loss = 3.2010e-06, Val Loss = 3.3831e-06, Time = 3.4056e-01, lr = 3.9063e-05
Epoch 400: Train Loss = 3.2016e-06, Val Loss = 3.3554e-06, Time = 3.0509e-01, lr = 7.6294e-08
Epoch 500: Train Loss = 3.2103e-06, Val Loss = 3.3023e-06, Time = 3.1395

([2.89820949035402e-06], 0)

In [13]:
# --- Configuration ---
k_folds = k_folds
n_epochs = 1000  
batch_size = 50 
learning_rate = 1e-2
print_every = 100
_precision = 4
random_state = random_state


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)


val_results = []
train_results = []

def get_fresh_model_components():
    f = FeluSigmoidMLP(dims=[2,10,10,10,2],lower_bound=-1, upper_bound=-0.1)
    g = GeluSigmoidMLP(dims=[4,10,10,10,2],lower_bound=0, upper_bound=1)
    model = FTNODE(f,g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            
            ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            u_func = lambda t: ui_expanded

            opt.zero_grad()
            
            dXi_pred = model_fold(ti, Xi, u_func)
            loss = loss_criteria(dXi, dXi_pred)
            
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                u_func = lambda t: ui_expanded

                dXi_pred = model_fold(ti, Xi, u_func)
                loss = loss_criteria(dXi, dXi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)
avg_best_val_losses, np.argmin(avg_best_val_losses)


--- FOLD 1/10 ---


Fold 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 2.0949e-02, Val Loss = 6.0622e-04, Time = 4.3543e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 3.1087e-04, Val Loss = 2.0064e-04, Time = 4.0106e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.8671e-04, Val Loss = 1.5371e-04, Time = 4.2019e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 1.4485e-04, Val Loss = 1.1008e-04, Time = 4.1686e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 1.1433e-04, Val Loss = 9.0172e-05, Time = 4.1787e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 9.7335e-05, Val Loss = 7.9276e-05, Time = 4.1225e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 4.3339e-06, Val Loss = 4.0529e-06, Time = 3.9724e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.4176e-06, Val Loss = 3.5031e-06, Time = 3.8696e-01, lr = 2.5000e-03
Epoch 300: Train Loss = 3.2747e-06, Val Loss = 3.3281e-06, Time = 4.4254e-01, lr = 1.9531e-05
Epoch 400: Train Loss = 3.2686e-06, Val Loss = 3.0356e-06, Time = 3.9018e-01, lr = 7.6294e-08
Epoch 500: Train Loss = 3.2769e-06, Val Loss = 3.1155e-06, Time = 3.8367

Fold 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.7171e-02, Val Loss = 7.9594e-04, Time = 4.1023e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 3.5317e-04, Val Loss = 2.5172e-04, Time = 3.7585e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.6657e-04, Val Loss = 1.2824e-04, Time = 4.0463e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 9.0832e-05, Val Loss = 7.0727e-05, Time = 4.1524e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 5.9332e-05, Val Loss = 5.0504e-05, Time = 3.9235e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 3.9325e-05, Val Loss = 3.4399e-05, Time = 4.2751e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 4.3351e-06, Val Loss = 4.7459e-06, Time = 4.1583e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.3847e-06, Val Loss = 3.6035e-06, Time = 4.0099e-01, lr = 1.2500e-03
Epoch 300: Train Loss = 3.2680e-06, Val Loss = 3.5224e-06, Time = 3.9437e-01, lr = 9.7656e-06
Epoch 400: Train Loss = 3.2710e-06, Val Loss = 3.8498e-06, Time = 3.9895e-01, lr = 1.9073e-08
Epoch 500: Train Loss = 3.2701e-06, Val Loss = 3.6511e-06, Time = 4.8335

...

Fold 9:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.7620e-02, Val Loss = 1.0949e-03, Time = 4.1738e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 7.6291e-04, Val Loss = 3.1853e-04, Time = 4.2788e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.4916e-04, Val Loss = 7.9254e-05, Time = 3.9696e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 6.7689e-05, Val Loss = 5.7657e-05, Time = 3.9441e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 5.5496e-05, Val Loss = 4.9426e-05, Time = 4.5807e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 4.7098e-05, Val Loss = 4.0359e-05, Time = 4.0555e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 5.2453e-06, Val Loss = 4.8851e-06, Time = 3.8638e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.4592e-06, Val Loss = 3.0112e-06, Time = 3.9649e-01, lr = 2.5000e-03
Epoch 300: Train Loss = 3.3694e-06, Val Loss = 3.0040e-06, Time = 4.0680e-01, lr = 1.9531e-05
Epoch 400: Train Loss = 3.3652e-06, Val Loss = 3.2519e-06, Time = 3.9799e-01, lr = 7.6294e-08
Epoch 500: Train Loss = 3.3730e-06, Val Loss = 3.1316e-06, Time = 4.0701

Fold 10:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.8863e-02, Val Loss = 1.2180e-03, Time = 4.5092e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 7.8469e-04, Val Loss = 2.7492e-04, Time = 3.8586e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.7091e-04, Val Loss = 9.3563e-05, Time = 4.2091e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 8.1399e-05, Val Loss = 6.4717e-05, Time = 5.0490e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 5.5427e-05, Val Loss = 4.2816e-05, Time = 4.5579e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 3.9852e-05, Val Loss = 3.3312e-05, Time = 4.5094e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.7240e-06, Val Loss = 4.1422e-06, Time = 4.1445e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.0779e-06, Val Loss = 3.5858e-06, Time = 3.9923e-01, lr = 6.2500e-04
Epoch 300: Train Loss = 3.0405e-06, Val Loss = 3.2039e-06, Time = 3.9988e-01, lr = 4.8828e-06
Epoch 400: Train Loss = 3.0501e-06, Val Loss = 3.1533e-06, Time = 4.0152e-01, lr = 1.9073e-08
Epoch 500: Train Loss = 3.0498e-06, Val Loss = 3.0892e-06, Time = 3.9301

([2.89820949035402e-06, 2.9250082974385804e-06], 0)

In [14]:
# --- Configuration ---
k_folds = k_folds
n_epochs = 1000  
batch_size = 50 
learning_rate = 1e-2
print_every = 100
_precision = 4
random_state = random_state


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)


val_results = []
train_results = []


def get_fresh_model_components():
    f = FeluSigmoidMLP(dims=[2,20,20,2],lower_bound=-1, upper_bound=-0.1)
    g = GeluSigmoidMLP(dims=[4,20,20,2],lower_bound=0, upper_bound=1)
    model = FTNODE(f,g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            
            ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            u_func = lambda t: ui_expanded

            opt.zero_grad()
            
            dXi_pred = model_fold(ti, Xi, u_func)
            loss = loss_criteria(dXi, dXi_pred)
            
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                u_func = lambda t: ui_expanded

                dXi_pred = model_fold(ti, Xi, u_func)
                loss = loss_criteria(dXi, dXi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)
avg_best_val_losses, np.argmin(avg_best_val_losses)


--- FOLD 1/10 ---


Fold 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.0866e-02, Val Loss = 1.7651e-04, Time = 5.6572e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.3064e-04, Val Loss = 7.8053e-05, Time = 5.0899e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 7.3780e-05, Val Loss = 5.4526e-05, Time = 5.3212e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 6.0480e-05, Val Loss = 4.8301e-05, Time = 5.5703e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 5.1768e-05, Val Loss = 3.8563e-05, Time = 5.9195e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 4.2949e-05, Val Loss = 3.0031e-05, Time = 5.1008e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 5.6701e-06, Val Loss = 5.6791e-06, Time = 4.9261e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.9522e-06, Val Loss = 2.7364e-06, Time = 4.9809e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 2.5872e-06, Val Loss = 2.4944e-06, Time = 5.4057e-01, lr = 3.1250e-04
Epoch 400: Train Loss = 2.5678e-06, Val Loss = 2.5702e-06, Time = 5.3966e-01, lr = 4.8828e-06
Epoch 500: Train Loss = 2.5704e-06, Val Loss = 2.5667e-06, Time = 5.7652

Fold 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.3523e-02, Val Loss = 3.1918e-04, Time = 5.4105e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.7900e-04, Val Loss = 1.2790e-04, Time = 5.5117e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 9.1693e-05, Val Loss = 8.1912e-05, Time = 5.0570e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 6.7505e-05, Val Loss = 6.3453e-05, Time = 5.6498e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 5.5754e-05, Val Loss = 5.4696e-05, Time = 5.1670e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 4.8538e-05, Val Loss = 4.9250e-05, Time = 5.3236e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 7.4547e-06, Val Loss = 8.5023e-06, Time = 5.6409e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.1241e-06, Val Loss = 3.3895e-06, Time = 5.7239e-01, lr = 5.0000e-03
Epoch 300: Train Loss = 3.0052e-06, Val Loss = 3.0778e-06, Time = 5.1980e-01, lr = 7.8125e-05
Epoch 400: Train Loss = 2.9922e-06, Val Loss = 3.1798e-06, Time = 5.5632e-01, lr = 6.1035e-07
Epoch 500: Train Loss = 2.9805e-06, Val Loss = 3.1261e-06, Time = 5.1789

...

Fold 9:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.0807e-02, Val Loss = 7.3395e-04, Time = 5.5707e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 3.2802e-04, Val Loss = 1.5904e-04, Time = 5.3805e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.0180e-04, Val Loss = 7.3862e-05, Time = 5.9033e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 5.8019e-05, Val Loss = 4.8103e-05, Time = 5.6935e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 4.2254e-05, Val Loss = 3.6621e-05, Time = 5.4628e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 3.2025e-05, Val Loss = 2.7971e-05, Time = 5.3488e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 4.6654e-06, Val Loss = 4.5209e-06, Time = 5.0154e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.9804e-06, Val Loss = 2.7337e-06, Time = 5.4179e-01, lr = 2.5000e-03
Epoch 300: Train Loss = 2.9120e-06, Val Loss = 2.6422e-06, Time = 5.0892e-01, lr = 1.9531e-05
Epoch 400: Train Loss = 2.9170e-06, Val Loss = 2.6505e-06, Time = 5.2087e-01, lr = 3.8147e-08
Epoch 500: Train Loss = 2.9075e-06, Val Loss = 2.6062e-06, Time = 4.8398

Fold 10:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.3804e-02, Val Loss = 4.8303e-04, Time = 5.8961e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 2.3495e-04, Val Loss = 1.3397e-04, Time = 5.4236e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 9.6234e-05, Val Loss = 6.5754e-05, Time = 5.1320e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 5.6420e-05, Val Loss = 4.1223e-05, Time = 5.0749e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 4.0534e-05, Val Loss = 3.3915e-05, Time = 5.1287e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 3.2116e-05, Val Loss = 2.8379e-05, Time = 5.0197e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 5.8741e-06, Val Loss = 5.4710e-06, Time = 5.7831e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.3178e-06, Val Loss = 3.4893e-06, Time = 4.9059e-01, lr = 2.5000e-03
Epoch 300: Train Loss = 3.0202e-06, Val Loss = 3.3464e-06, Time = 5.3448e-01, lr = 7.8125e-05
Epoch 400: Train Loss = 3.0218e-06, Val Loss = 3.0465e-06, Time = 5.7409e-01, lr = 3.0518e-07
Epoch 500: Train Loss = 3.0294e-06, Val Loss = 3.0446e-06, Time = 5.3491

([2.89820949035402e-06, 2.9250082974385804e-06, 2.6281525194349344e-06], 2)

In [15]:
# --- Configuration ---
k_folds = k_folds
n_epochs = 1000  
batch_size = 50 
learning_rate = 1e-2
print_every = 100
_precision = 4
random_state = random_state


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)


val_results = []
train_results = []

def get_fresh_model_components():
    f = FeluSigmoidMLP(dims=[2,50,50,2],lower_bound=-1, upper_bound=-0.1)
    g = GeluSigmoidMLP(dims=[4,50,50,2],lower_bound=0, upper_bound=1)
    model = FTNODE(f,g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            
            ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            u_func = lambda t: ui_expanded

            opt.zero_grad()
            
            dXi_pred = model_fold(ti, Xi, u_func)
            loss = loss_criteria(dXi, dXi_pred)
            
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                u_func = lambda t: ui_expanded

                dXi_pred = model_fold(ti, Xi, u_func)
                loss = loss_criteria(dXi, dXi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)
avg_best_val_losses, np.argmin(avg_best_val_losses)


--- FOLD 1/10 ---


Fold 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 5.4196e-03, Val Loss = 1.7383e-04, Time = 9.1536e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.1704e-04, Val Loss = 5.6120e-05, Time = 8.6258e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 5.0297e-05, Val Loss = 2.9120e-05, Time = 8.8147e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 2.9380e-05, Val Loss = 1.8466e-05, Time = 8.7247e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 1.7508e-05, Val Loss = 1.4057e-05, Time = 8.5857e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 1.4417e-05, Val Loss = 1.2440e-05, Time = 8.6029e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 4.1772e-06, Val Loss = 3.9754e-06, Time = 9.3729e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.9117e-06, Val Loss = 2.8970e-06, Time = 1.3676e+00, lr = 6.2500e-04
Epoch 300: Train Loss = 2.8708e-06, Val Loss = 2.8524e-06, Time = 8.4180e-01, lr = 1.2207e-06
Epoch 400: Train Loss = 2.8874e-06, Val Loss = 2.8473e-06, Time = 8.2374e-01, lr = 1.9073e-08
Epoch 500: Train Loss = 2.8697e-06, Val Loss = 2.7993e-06, Time = 8.3674

Fold 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 6.0297e-03, Val Loss = 1.3718e-04, Time = 9.2487e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 8.0322e-05, Val Loss = 5.5156e-05, Time = 9.2720e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 4.1725e-05, Val Loss = 3.6783e-05, Time = 9.0176e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 2.9373e-05, Val Loss = 2.7375e-05, Time = 8.6180e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 2.2242e-05, Val Loss = 2.1947e-05, Time = 9.5098e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 1.7940e-05, Val Loss = 1.9054e-05, Time = 9.4211e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.9712e-06, Val Loss = 4.1715e-06, Time = 1.0453e+00, lr = 1.0000e-02
Epoch 200: Train Loss = 3.0109e-06, Val Loss = 2.9275e-06, Time = 8.5735e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 2.4399e-06, Val Loss = 2.4861e-06, Time = 8.4195e-01, lr = 7.8125e-05
Epoch 400: Train Loss = 2.4409e-06, Val Loss = 2.4843e-06, Time = 8.1622e-01, lr = 3.0518e-07
Epoch 500: Train Loss = 2.4276e-06, Val Loss = 2.4976e-06, Time = 8.1577

...

Fold 9:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 6.3130e-03, Val Loss = 1.5247e-04, Time = 9.3621e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 9.3286e-05, Val Loss = 7.1927e-05, Time = 8.5071e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 5.5473e-05, Val Loss = 4.6603e-05, Time = 8.3663e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 3.9728e-05, Val Loss = 3.4316e-05, Time = 9.0028e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 2.7620e-05, Val Loss = 2.1980e-05, Time = 8.6992e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 1.9371e-05, Val Loss = 1.6113e-05, Time = 8.1784e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 4.0273e-06, Val Loss = 3.7456e-06, Time = 8.3276e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.8430e-06, Val Loss = 2.7897e-06, Time = 9.2671e-01, lr = 2.5000e-03
Epoch 300: Train Loss = 2.7942e-06, Val Loss = 2.5536e-06, Time = 8.0376e-01, lr = 4.8828e-06
Epoch 400: Train Loss = 2.7761e-06, Val Loss = 2.5268e-06, Time = 7.8465e-01, lr = 1.9073e-08
Epoch 500: Train Loss = 2.7775e-06, Val Loss = 2.4671e-06, Time = 7.8392

Fold 10:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 5.7217e-03, Val Loss = 1.1162e-04, Time = 1.0887e+00, lr = 1.0000e-02
Epoch 1: Train Loss = 7.9824e-05, Val Loss = 4.8020e-05, Time = 8.4195e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 4.3582e-05, Val Loss = 3.1462e-05, Time = 8.2953e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 2.9808e-05, Val Loss = 2.3123e-05, Time = 8.5260e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 2.0977e-05, Val Loss = 1.7908e-05, Time = 8.8936e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 1.6831e-05, Val Loss = 1.5085e-05, Time = 8.8777e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.5305e-06, Val Loss = 3.8560e-06, Time = 8.6816e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.0185e-06, Val Loss = 3.1797e-06, Time = 8.4390e-01, lr = 1.5625e-04
Epoch 300: Train Loss = 3.0230e-06, Val Loss = 3.1701e-06, Time = 8.5434e-01, lr = 3.0518e-07
Epoch 400: Train Loss = 3.0237e-06, Val Loss = 2.8626e-06, Time = 8.3709e-01, lr = 1.9073e-08
Epoch 500: Train Loss = 3.0128e-06, Val Loss = 3.0973e-06, Time = 8.8059

([2.89820949035402e-06,
  2.9250082974385804e-06,
  2.6281525194349344e-06,
  2.573639753933321e-06],
 3)

In [16]:
# --- Configuration ---
k_folds = k_folds
n_epochs = 1000  
batch_size = 50 
learning_rate = 1e-2
print_every = 100
_precision = 4
random_state = random_state


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)


val_results = []
train_results = []

def get_fresh_model_components():
    f = FeluSigmoidMLP(dims=[2,20,20,20,2],lower_bound=-1, upper_bound=-0.1)
    g = GeluSigmoidMLP(dims=[4,20,20,20,2],lower_bound=0, upper_bound=1)
    model = FTNODE(f,g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            
            ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            u_func = lambda t: ui_expanded

            opt.zero_grad()
            
            dXi_pred = model_fold(ti, Xi, u_func)
            loss = loss_criteria(dXi, dXi_pred)
            
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                u_func = lambda t: ui_expanded

                dXi_pred = model_fold(ti, Xi, u_func)
                loss = loss_criteria(dXi, dXi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)
avg_best_val_losses, np.argmin(avg_best_val_losses)


--- FOLD 1/10 ---


Fold 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.0922e-02, Val Loss = 2.3318e-04, Time = 7.5620e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.6855e-04, Val Loss = 8.5012e-05, Time = 6.7863e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 7.1013e-05, Val Loss = 4.4907e-05, Time = 7.5117e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 5.2704e-05, Val Loss = 3.4536e-05, Time = 7.2294e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 3.8651e-05, Val Loss = 2.4257e-05, Time = 7.5798e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 2.6017e-05, Val Loss = 1.7206e-05, Time = 7.9110e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.2649e-06, Val Loss = 3.2248e-06, Time = 7.1595e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.4422e-06, Val Loss = 2.3639e-06, Time = 6.9839e-01, lr = 2.5000e-03
Epoch 300: Train Loss = 2.3204e-06, Val Loss = 2.2673e-06, Time = 7.2576e-01, lr = 9.7656e-06
Epoch 400: Train Loss = 2.3193e-06, Val Loss = 2.0760e-06, Time = 7.1864e-01, lr = 1.9073e-08
Epoch 500: Train Loss = 2.3143e-06, Val Loss = 2.0724e-06, Time = 7.3469

Fold 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.1923e-02, Val Loss = 4.4775e-04, Time = 9.8721e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 2.6746e-04, Val Loss = 1.5689e-04, Time = 7.2488e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.0628e-04, Val Loss = 1.0100e-04, Time = 7.2438e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 7.4004e-05, Val Loss = 7.4314e-05, Time = 6.8847e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 6.4919e-05, Val Loss = 6.8284e-05, Time = 6.9862e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 5.7210e-05, Val Loss = 6.0012e-05, Time = 6.8829e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 4.2915e-06, Val Loss = 5.1212e-06, Time = 6.3885e-01, lr = 5.0000e-03
Epoch 200: Train Loss = 3.5699e-06, Val Loss = 3.9784e-06, Time = 6.4575e-01, lr = 6.2500e-04
Epoch 300: Train Loss = 3.4062e-06, Val Loss = 3.8824e-06, Time = 6.7776e-01, lr = 1.9531e-05
Epoch 400: Train Loss = 3.3991e-06, Val Loss = 3.8023e-06, Time = 7.0597e-01, lr = 3.8147e-08
Epoch 500: Train Loss = 3.3978e-06, Val Loss = 3.7011e-06, Time = 6.5723

...

Fold 9:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.1724e-02, Val Loss = 1.5347e-04, Time = 7.4486e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.2250e-04, Val Loss = 7.9011e-05, Time = 7.0164e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 7.4199e-05, Val Loss = 6.3267e-05, Time = 6.6034e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 6.0547e-05, Val Loss = 5.3814e-05, Time = 6.8048e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 4.9051e-05, Val Loss = 4.2176e-05, Time = 7.3301e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 3.8288e-05, Val Loss = 3.1073e-05, Time = 6.9126e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.6089e-06, Val Loss = 3.2776e-06, Time = 9.5228e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 3.2927e-06, Val Loss = 3.0159e-06, Time = 6.4239e-01, lr = 7.8125e-05
Epoch 300: Train Loss = 3.2970e-06, Val Loss = 2.9131e-06, Time = 6.6648e-01, lr = 3.0518e-07
Epoch 400: Train Loss = 3.2912e-06, Val Loss = 2.8918e-06, Time = 6.5741e-01, lr = 1.9073e-08
Epoch 500: Train Loss = 3.3019e-06, Val Loss = 3.0250e-06, Time = 6.8671

Fold 10:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.2050e-02, Val Loss = 2.2426e-04, Time = 7.4595e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 1.6439e-04, Val Loss = 8.6663e-05, Time = 6.9871e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 6.3376e-05, Val Loss = 4.4789e-05, Time = 6.6000e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 4.1195e-05, Val Loss = 3.1304e-05, Time = 6.5794e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 2.7085e-05, Val Loss = 2.0553e-05, Time = 6.6156e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 1.8345e-05, Val Loss = 1.5576e-05, Time = 6.7425e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.3655e-06, Val Loss = 3.3884e-06, Time = 6.6128e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.6164e-06, Val Loss = 2.7486e-06, Time = 6.6441e-01, lr = 1.2500e-03
Epoch 300: Train Loss = 2.4885e-06, Val Loss = 2.6734e-06, Time = 6.6304e-01, lr = 3.9063e-05
Epoch 400: Train Loss = 2.4856e-06, Val Loss = 2.6260e-06, Time = 7.0585e-01, lr = 1.5259e-07
Epoch 500: Train Loss = 2.4823e-06, Val Loss = 2.7246e-06, Time = 6.4213

([2.89820949035402e-06,
  2.9250082974385804e-06,
  2.6281525194349344e-06,
  2.573639753933321e-06,
  2.4570398473144904e-06],
 4)

In [17]:
# --- Configuration ---
k_folds = k_folds
n_epochs = 1000  
batch_size = 50 
learning_rate = 1e-2
print_every = 100
_precision = 4
random_state = random_state


kfold = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)


val_results = []
train_results = []

def get_fresh_model_components():
    f = FeluSigmoidMLP(dims=[2,10,10,10,10,2],lower_bound=-1, upper_bound=-0.1)
    g = GeluSigmoidMLP(dims=[4,10,10,10,10,2],lower_bound=0, upper_bound=1)
    model = FTNODE(f,g).to(device)
    return f, g, model 

# ==========================================
# K-Fold Loop
# ==========================================
for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
    print(f'\n--- FOLD {fold+1}/{k_folds} ---')

    # 1. Re-initialize Model & Optimizer for this fold
    f_fold, g_fold, model_fold = get_fresh_model_components()
    model_fold.train() 
    
    loss_criteria = nn.MSELoss()
    

    opt = torch.optim.Adam(
        list(f_fold.parameters()) + list(g_fold.parameters()), 
        lr=learning_rate
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=0.5, patience=10
    )

    # 2. Create DataLoaders for this fold
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader = DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
    valloader = DataLoader(dataset, batch_size=batch_size, sampler=val_subsampler)

    # 3. Training & Validation Loop
    best_val_loss = float('inf')
    fold_losses = []

    for epoch in tqdm(range(n_epochs), desc=f"Fold {fold+1}"):
        t1 = time.time()
        
        # --- TRAINING ---
        model_fold.train()
        train_loss = 0.0
        
        for batch_idx, (dXi, Xi, ti, ui) in enumerate(trainloader):
            
            ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
            u_func = lambda t: ui_expanded

            opt.zero_grad()
            
            dXi_pred = model_fold(ti, Xi, u_func)
            loss = loss_criteria(dXi, dXi_pred)
            
            loss.backward()
            opt.step()
            
            train_loss += loss.item()
        
        train_loss /= len(trainloader)

        # --- VALIDATION ---
        model_fold.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_idx, (dXi, Xi, ti, ui) in enumerate(valloader):
                ui_expanded = ui.unsqueeze(dim=1).expand(Xi.shape)
                u_func = lambda t: ui_expanded

                dXi_pred = model_fold(ti, Xi, u_func)
                loss = loss_criteria(dXi, dXi_pred)
                val_loss += loss.item()
        
        val_loss /= len(valloader)
        
        # --- SCHEDULER & LOGGING ---
        epoch_time = time.time() - t1
        
        # Step scheduler based on VALIDATION loss (standard practice)
        scheduler.step(val_loss) 
        cur_lr = opt.param_groups[0]['lr']

        if epoch <= 5 or epoch % print_every == 0 or epoch == n_epochs - 1:
            print(
                f"Epoch {epoch}: "
                f"Train Loss = {train_loss:.{_precision}e}, "
                f"Val Loss = {val_loss:.{_precision}e}, "
                f"Time = {epoch_time:.{_precision}e}, "
                f"lr = {cur_lr:.{_precision}e}"
            )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_train_loss = train_loss
            best_fold = fold
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.{_precision}e}")
    val_results.append(best_val_loss)
    train_results.append(best_val_train_loss)

# --- SUMMARY ---
print("\nK-Fold Cross Validation Results:")
avg_loss = np.mean(val_results)
avg_train_loss = np.mean(train_results)
print(f"Average Best Validation Loss: {avg_loss:.{_precision}e}")
avg_best_val_losses.append(avg_loss)
avg_best_train_losses.append(avg_train_loss)
avg_best_val_losses, np.argmin(avg_best_val_losses)


--- FOLD 1/10 ---


Fold 1:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.9020e-02, Val Loss = 3.5579e-04, Time = 4.9912e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 3.0302e-04, Val Loss = 2.0677e-04, Time = 4.8020e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.7213e-04, Val Loss = 1.1532e-04, Time = 4.7167e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 1.0624e-04, Val Loss = 6.3806e-05, Time = 5.5257e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 7.3797e-05, Val Loss = 5.4178e-05, Time = 4.7371e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 6.2628e-05, Val Loss = 4.3269e-05, Time = 5.0994e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.4615e-06, Val Loss = 3.2228e-06, Time = 4.8743e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.9631e-06, Val Loss = 2.8594e-06, Time = 5.4071e-01, lr = 3.1250e-04
Epoch 300: Train Loss = 2.9305e-06, Val Loss = 2.9016e-06, Time = 4.8820e-01, lr = 6.1035e-07
Epoch 400: Train Loss = 2.9339e-06, Val Loss = 2.9056e-06, Time = 5.3647e-01, lr = 1.9073e-08
Epoch 500: Train Loss = 2.9375e-06, Val Loss = 2.8113e-06, Time = 4.9125

Fold 2:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.9404e-02, Val Loss = 7.7612e-04, Time = 6.0367e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 3.3789e-04, Val Loss = 2.1451e-04, Time = 7.9884e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.5228e-04, Val Loss = 1.2298e-04, Time = 5.2292e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 9.6475e-05, Val Loss = 9.5572e-05, Time = 4.8903e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 7.5955e-05, Val Loss = 7.0610e-05, Time = 4.8888e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 5.6019e-05, Val Loss = 5.0226e-05, Time = 4.6960e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.8513e-06, Val Loss = 4.7765e-06, Time = 4.7338e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.7556e-06, Val Loss = 2.9272e-06, Time = 5.3798e-01, lr = 1.0000e-02
Epoch 300: Train Loss = 2.6036e-06, Val Loss = 2.9797e-06, Time = 4.8186e-01, lr = 3.9063e-05
Epoch 400: Train Loss = 2.5835e-06, Val Loss = 2.7560e-06, Time = 4.9380e-01, lr = 1.5259e-07
Epoch 500: Train Loss = 2.5783e-06, Val Loss = 2.6839e-06, Time = 4.8218

...

Fold 9:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 1.9611e-02, Val Loss = 7.1535e-04, Time = 5.1120e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 4.2924e-04, Val Loss = 2.5353e-04, Time = 5.6428e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.4903e-04, Val Loss = 8.7609e-05, Time = 5.2029e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 7.4124e-05, Val Loss = 6.1206e-05, Time = 4.8842e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 5.5680e-05, Val Loss = 4.6652e-05, Time = 5.0722e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 4.4434e-05, Val Loss = 3.8628e-05, Time = 5.0031e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.5945e-06, Val Loss = 3.3134e-06, Time = 4.9422e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.6493e-06, Val Loss = 2.2351e-06, Time = 4.9122e-01, lr = 2.5000e-03
Epoch 300: Train Loss = 2.5211e-06, Val Loss = 2.1291e-06, Time = 4.9470e-01, lr = 9.7656e-06
Epoch 400: Train Loss = 2.5327e-06, Val Loss = 2.1382e-06, Time = 4.8372e-01, lr = 1.9073e-08
Epoch 500: Train Loss = 2.5298e-06, Val Loss = 2.0957e-06, Time = 4.8479

Fold 10:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0: Train Loss = 2.2436e-02, Val Loss = 6.1943e-04, Time = 5.3482e-01, lr = 1.0000e-02
Epoch 1: Train Loss = 2.9402e-04, Val Loss = 1.9061e-04, Time = 4.9211e-01, lr = 1.0000e-02
Epoch 2: Train Loss = 1.4902e-04, Val Loss = 1.1817e-04, Time = 4.8347e-01, lr = 1.0000e-02
Epoch 3: Train Loss = 1.0695e-04, Val Loss = 9.0187e-05, Time = 6.1042e-01, lr = 1.0000e-02
Epoch 4: Train Loss = 8.5585e-05, Val Loss = 7.5844e-05, Time = 4.7958e-01, lr = 1.0000e-02
Epoch 5: Train Loss = 7.4517e-05, Val Loss = 6.5500e-05, Time = 4.9161e-01, lr = 1.0000e-02
Epoch 100: Train Loss = 3.5227e-06, Val Loss = 3.4620e-06, Time = 4.9276e-01, lr = 1.0000e-02
Epoch 200: Train Loss = 2.5814e-06, Val Loss = 2.6716e-06, Time = 4.8589e-01, lr = 6.2500e-04
Epoch 300: Train Loss = 2.5144e-06, Val Loss = 2.7807e-06, Time = 5.3636e-01, lr = 4.8828e-06
Epoch 400: Train Loss = 2.5158e-06, Val Loss = 2.7949e-06, Time = 4.9016e-01, lr = 1.9073e-08
Epoch 500: Train Loss = 2.5035e-06, Val Loss = 2.5868e-06, Time = 4.7409

([2.89820949035402e-06,
  2.9250082974385804e-06,
  2.6281525194349344e-06,
  2.573639753933321e-06,
  2.4570398473144904e-06,
  2.5669246397797e-06],
 4)

In [18]:
avg_best_val_losses, np.argmin(avg_best_val_losses)

([2.89820949035402e-06,
  2.9250082974385804e-06,
  2.6281525194349344e-06,
  2.573639753933321e-06,
  2.4570398473144904e-06,
  2.5669246397797e-06],
 4)